In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# 1. Load the breast cancer dataset and create train (60%), validation (20%), and test (20%) splits
data = load_breast_cancer()
X, y = data.data, data.target

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

# 2. Train Logistic Regression as the baseline (unscaled) and record validation accuracy
baseline_model = LogisticRegression(max_iter=5000, random_state=42)
baseline_model.fit(X_train, y_train)
baseline_score = accuracy_score(y_val, baseline_model.predict(X_val))

# 3 & 4. Create a pipeline with StandardScaler + Logistic Regression and run 5-fold CV
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

# 5. Use GridSearchCV to tune hyperparameter C and compare results
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(pipe, param_grid=param_grid, cv=cv, scoring="accuracy")
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
tuned_val_score = accuracy_score(y_val, best_model.predict(X_val))
final_test_score = accuracy_score(y_test, best_model.predict(X_test))

# Display Submission Comparison Table
results_df = pd.DataFrame({
    "Stage": [
        "Baseline (Unscaled Val)",
        "5-Fold CV Mean (Scaled Train)",
        f"Tuned GridSearchCV (Val, C={grid.best_params_['model__C']})",
        "Final Tuned Model (Unseen Test)"
    ],
    "Accuracy": [
        round(baseline_score, 4),
        f"{cv_mean:.4f} (± {cv_std:.4f})",
        round(tuned_val_score, 4),
        round(final_test_score, 4)
    ]
})

print(results_df.to_string(index=False))


                          Stage          Accuracy
        Baseline (Unscaled Val)            0.9474
  5-Fold CV Mean (Scaled Train) 0.9649 (± 0.0218)
Tuned GridSearchCV (Val, C=0.1)            0.9737
Final Tuned Model (Unseen Test)            0.9737
